# Assignment 02: Autograd Deep Dive

**Module:** 03 — PyTorch Fundamentals  
**Estimated Time:** 5-7 hours

---

## Learning Objectives

- Compute forward and backward passes by hand and verify against PyTorch autograd
- Implement custom `torch.autograd.Function` (Swish, STE, Clamp)
- Understand gradient accumulation and its role in large-batch training
- Implement gradient checking via numerical differentiation
- Implement gradient clipping from scratch
- Explore higher-order gradients, `detach()`, `no_grad()`, and `retain_graph`

### Key Concepts

- **Computational graph:** Directed acyclic graph of operations, each node has a `grad_fn`
- **Backward pass:** Traverse graph in reverse, applying chain rule at each node
- **Gradient accumulation:** Gradients are **added** to `.grad`, not replaced
- **Gradient clipping:** Scale gradients if total norm exceeds threshold:
  $$\mathbf{g} \leftarrow \frac{\text{max\_norm}}{\|\mathbf{g}\|} \cdot \mathbf{g} \quad \text{if } \|\mathbf{g}\| > \text{max\_norm}$$

In [ ]:
import sys
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

sys.path.insert(0, '../../')
from shared_utils.common import set_seed

set_seed(42)
print(f'PyTorch version: {torch.__version__}')
print('Setup complete.')

---
## Part 1: Manual Backward Pass (60 min)

### Exercise 1.1: Linear Model

Compute $z = x \mathbf{W} + b$, $L = \text{mean}((z - t)^2)$ by hand, then verify.

In [ ]:
# --- 1.1: Linear model ---
w = torch.tensor([[1.0, 2.0], [3.0, 4.0]], requires_grad=True)
b = torch.tensor([0.5, -0.5], requires_grad=True)
x = torch.tensor([1.0, -1.0])
target = torch.tensor([2.0, 0.0])

# Forward
z = x @ w + b
loss = ((z - target) ** 2).mean()

# YOUR HAND COMPUTATION HERE:
# z = ?
# loss = ?
# dloss/dw = ?
# dloss/db = ?

loss.backward()

print(f'z = {z.detach()}')
print(f'loss = {loss.item():.6f}')
print(f'w.grad =\n{w.grad}')
print(f'b.grad = {b.grad}')
# Compare to your hand-computed values

### Exercise 1.2: Non-Linear Computation

In [ ]:
# --- 1.2: Non-linear ---
a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(3.0, requires_grad=True)

c = a * b
d = torch.sin(c)
e = d ** 2
f = e + a
loss = f * b

# YOUR HAND COMPUTATION:
# c = 6.0, d = sin(6), e = sin^2(6), f = sin^2(6)+2, loss = (sin^2(6)+2)*3
# dloss/da = ?
# dloss/db = ?

loss.backward()
print(f'loss = {loss.item():.6f}')
print(f'dloss/da = {a.grad.item():.6f}')
print(f'dloss/db = {b.grad.item():.6f}')

### Exercise 1.3: Branching Graph

In [ ]:
# --- 1.3: Branching ---
x = torch.tensor(3.0, requires_grad=True)
a = x ** 2
b = x ** 3
loss = a + b

# dloss/dx = 2x + 3x^2 = 6 + 27 = 33
# Gradients from both branches are SUMMED at x

loss.backward()
print(f'dloss/dx = {x.grad.item():.1f} (expected: 33.0)')
assert abs(x.grad.item() - 33.0) < 1e-5

# YOUR CODE HERE: Add third branch c = exp(x), recompute
pass

---
## Part 2: Custom Autograd Functions (90 min)

### Exercise 2.1: Swish Activation

$\text{swish}(x) = x \cdot \sigma(x)$

$\text{swish}'(x) = \sigma(x)(1 + x(1 - \sigma(x)))$

In [ ]:
class SwishFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        """Forward: swish(x) = x * sigmoid(x).
        Save what's needed for backward using ctx.save_for_backward.
        """
        # YOUR CODE HERE
        pass

    @staticmethod
    def backward(ctx, grad_output):
        """Backward: swish'(x) = sigmoid(x) * (1 + x * (1 - sigmoid(x)))."""
        # YOUR CODE HERE
        pass

swish = SwishFunction.apply

In [ ]:
# --- Test 2.1 ---
x_test = torch.randn(5, dtype=torch.float64, requires_grad=True)

# Gradcheck (requires float64)
assert torch.autograd.gradcheck(swish, x_test), 'Swish gradcheck failed!'

# Compare to naive implementation
x_compare = torch.randn(10, requires_grad=True)
y_custom = swish(x_compare)
y_naive = x_compare * torch.sigmoid(x_compare)
assert torch.allclose(y_custom, y_naive, atol=1e-6), 'Swish output mismatch'

print('Exercise 2.1 passed!')

### Exercise 2.2: Straight-Through Estimator (STE)

In [ ]:
class STEFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        """Forward: return sign(x)."""
        # YOUR CODE HERE
        pass

    @staticmethod
    def backward(ctx, grad_output):
        """Backward: pass grad through unchanged (identity)."""
        # YOUR CODE HERE
        pass

ste_sign = STEFunction.apply

In [ ]:
# --- Test 2.2 ---
x_ste = torch.randn(5, requires_grad=True)
y_ste = ste_sign(x_ste)
loss = y_ste.sum()
loss.backward()

print(f'Input:    {x_ste.data}')
print(f'Output:   {y_ste.data}')  # Should be +1 or -1
print(f'Gradient: {x_ste.grad}')   # Should be all 1s (identity)
assert torch.all(x_ste.grad == 1.0), 'STE gradient should be all 1s'
print('Exercise 2.2 passed!')

# Use case: quantization, binary neural networks

### Exercise 2.3: Custom Clamp Function

In [ ]:
class ClampFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, min_val, max_val):
        """Forward: clamp(x, min_val, max_val).
        Save mask of unclamped values for backward.
        """
        # YOUR CODE HERE
        pass

    @staticmethod
    def backward(ctx, grad_output):
        """Backward: gradient is 1 where unclamped, 0 where clamped."""
        # YOUR CODE HERE
        pass

custom_clamp = ClampFunction.apply

In [ ]:
# --- Test 2.3 ---
x_clamp = torch.randn(10, dtype=torch.float64, requires_grad=True)
assert torch.autograd.gradcheck(
    lambda x: custom_clamp(x, -0.5, 0.5), x_clamp
), 'Clamp gradcheck failed!'

# Compare to torch.clamp
x_c = torch.randn(10, requires_grad=True)
y_custom = custom_clamp(x_c, -0.5, 0.5)
y_builtin = torch.clamp(x_c, -0.5, 0.5)
assert torch.allclose(y_custom, y_builtin), 'Clamp output mismatch'
print('Exercise 2.3 passed!')

---
## Part 3: Gradient Accumulation Experiments (45 min)

### Exercise 3.1: Observing Accumulation

In [ ]:
# --- 3.1: Observe gradient accumulation ---
w = torch.tensor(1.0, requires_grad=True)

y1 = w * 2
y1.backward()
print(f'After y1=w*2: w.grad = {w.grad}')  # Expected: 2

y2 = w * 3
y2.backward()
print(f'After y2=w*3: w.grad = {w.grad}')  # Expected: 2+3=5

y3 = w * 5
y3.backward()
print(f'After y3=w*5: w.grad = {w.grad}')  # Expected: 2+3+5=10

# YOUR CODE HERE: Zero gradient and verify
pass

### Exercise 3.2: Simulating Large Batch Training

In [ ]:
# --- 3.2: Gradient accumulation for large effective batch ---
# YOUR CODE HERE
# 1. Create synthetic data and DataLoader with batch_size=8
# 2. Train with accumulation_steps=4 (effective batch=32)
# 3. Scale loss by 1/accumulation_steps
# 4. Compare to true batch_size=32 training
#
# Use a simple linear model: nn.Linear(10, 1)
# Show that final weights are similar
pass

### Exercise 3.3: When Accumulation Goes Wrong

In [ ]:
# --- 3.3: Forgetting zero_grad ---
# YOUR CODE HERE
# Train two models: one with proper zero_grad, one without
# Plot loss curves to show divergence
pass

---
## Part 4: Gradient Checking (45 min)

In [ ]:
def numerical_gradient(func, x: torch.Tensor, eps: float = 1e-5) -> torch.Tensor:
    """Compute numerical gradient using central differences.

    Args:
        func: function taking a tensor, returning a scalar
        x: point at which to compute gradient (1D tensor)
        eps: finite difference step

    Returns:
        Tensor of same shape as x with numerical gradients
    """
    # YOUR CODE HERE
    # For each element i of x:
    #   x_plus = x.clone(); x_plus[i] += eps
    #   x_minus = x.clone(); x_minus[i] -= eps
    #   grad[i] = (func(x_plus) - func(x_minus)) / (2 * eps)
    pass

In [ ]:
# --- Test Part 4 ---
# Simple: f(x) = x^2 at x=3 -> grad=6
x_t = torch.tensor([3.0])
ng = numerical_gradient(lambda x: (x**2).sum(), x_t)
print(f'x^2 at x=3: numerical grad = {ng.item():.6f} (expected 6.0)')

# Complex function
def complex_func(x):
    return (x ** 3 + torch.sin(x * 2)).sum()

x_test = torch.randn(5, requires_grad=True)
y = complex_func(x_test)
y.backward()
analytical = x_test.grad.clone()
numerical = numerical_gradient(complex_func, x_test.detach())

max_diff = (analytical - numerical).abs().max().item()
print(f'Max difference: {max_diff:.2e}')
assert max_diff < 1e-4, 'Gradient check failed!'
print('Part 4 passed!')

---
## Part 5: Gradient Clipping from Scratch (30 min)

In [ ]:
def clip_grad_norm(parameters, max_norm: float) -> float:
    """Clip gradients by total L2 norm.

    If total_norm > max_norm, scale all gradients by (max_norm / total_norm).

    Args:
        parameters: iterable of tensors with .grad
        max_norm: maximum allowed total gradient norm

    Returns:
        Total gradient norm before clipping
    """
    # YOUR CODE HERE
    # 1. Compute total_norm = sqrt(sum of squared grad norms)
    # 2. If total_norm > max_norm, scale each grad
    pass


def clip_grad_value(parameters, clip_value: float):
    """Clamp each gradient element to [-clip_value, clip_value]."""
    # YOUR CODE HERE
    pass

In [ ]:
# --- Test Part 5 ---
# Create a model and compute gradients
model = nn.Sequential(nn.Linear(10, 50), nn.ReLU(), nn.Linear(50, 1))
x = torch.randn(4, 10)
y = model(x).sum()
y.backward()

# Test clip_grad_norm
import copy
model_copy = copy.deepcopy(model)
# Apply your clipping
total_norm = clip_grad_norm(model.parameters(), max_norm=1.0)
# Apply PyTorch's clipping
torch.nn.utils.clip_grad_norm_(model_copy.parameters(), max_norm=1.0)

for (n1, p1), (n2, p2) in zip(model.named_parameters(), model_copy.named_parameters()):
    if p1.grad is not None:
        diff = (p1.grad - p2.grad).abs().max().item()
        print(f'{n1}: max grad diff = {diff:.2e}')
        assert diff < 1e-5, f'clip_grad_norm mismatch for {n1}'

print('Part 5 passed!')

---
## Part 6: Visualizing the Computational Graph (30 min)

In [ ]:
# --- Part 6: Computational graph visualization ---
# Option A: Use torchviz if installed
try:
    from torchviz import make_dot
    
    # Simple 2-layer network
    model_viz = nn.Sequential(nn.Linear(10, 5), nn.ReLU(), nn.Linear(5, 1))
    x_viz = torch.randn(1, 10)
    y_viz = model_viz(x_viz)
    dot = make_dot(y_viz, params=dict(model_viz.named_parameters()))
    dot.render('computational_graph', format='png', cleanup=True)
    print('Graph saved as computational_graph.png')
except ImportError:
    print('torchviz not installed. Draw ASCII graphs manually.')
    # YOUR ASCII GRAPH HERE for the computation in Part 1.2

---
## Part 7: Advanced Autograd Concepts (45 min)

### Exercise 7.1: Higher-Order Gradients

In [ ]:
# --- 7.1: Higher-order gradients ---
x = torch.tensor(3.0, requires_grad=True)
y = x ** 3  # dy/dx = 3x^2 = 27, d2y/dx2 = 6x = 18

# First derivative (create_graph=True to allow second derivative)
grad_y = torch.autograd.grad(y, x, create_graph=True)[0]
print(f'dy/dx = {grad_y.item():.1f} (expected 27.0)')

# Second derivative
grad_grad_y = torch.autograd.grad(grad_y, x, create_graph=True)[0]
print(f'd2y/dx2 = {grad_grad_y.item():.1f} (expected 18.0)')

# YOUR CODE HERE: Third derivative (expected: 6.0)
pass

# YOUR CODE HERE: Newton's method for f(x) = (x-3)^4, starting from x=0
pass

**What does `create_graph=True` do?**

*YOUR ANSWER HERE*

### Exercise 7.2: detach() vs no_grad()

In [ ]:
# --- 7.2: detach vs no_grad ---
x = torch.randn(3, requires_grad=True)

# 1. Inside no_grad() block
with torch.no_grad():
    y = x * 2
print(f'no_grad: y.requires_grad={y.requires_grad}, grad_fn={y.grad_fn}')

# 2. Using detach()
y2 = (x * 2)
z = y2.detach()
print(f'detach: z.requires_grad={z.requires_grad}, grad_fn={z.grad_fn}')

# 3. Does x get gradient through detached z?
w = z * 3
w.sum().backward()
print(f'x.grad after backward through detached: {x.grad}')

# YOUR CODE HERE: Summarize when to use no_grad vs detach

### Exercise 7.3: retain_graph

In [ ]:
# --- 7.3: retain_graph ---
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2

# First backward
y.backward(retain_graph=True)
print(f'First backward: x.grad = {x.grad}')

# Second backward (would fail without retain_graph)
y.backward()
print(f'Second backward: x.grad = {x.grad}')  # Accumulated!

# YOUR ANSWER: Why is retain_graph expensive? When is it necessary?

---
## Key Insights

Summarize what you learned about how autograd works internally:

*YOUR SUMMARY HERE*